In [5]:
# Alright so we have created our walk-forward validation using a 20 day rolling window
# where the averaged squared returns act as our realized volatility proxy - which we use
# to see if knowing this, allows us to predict better
# I've gone ahead and refactored run_fold and placed the previous functions we made in 01_baseline
# into src as reusable definitions
# so without further ado lets designate a shared data set for the regression tests
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


In [6]:
## Generate dummy returns
import numpy as np
import pandas as pd

np.random.seed(42)
dates = pd.bdate_range( start="2019-01-01", end="2025-12-31")
n = len(dates)
# Approximate daily stock-return assumptions
daily_drift = 0.0003
daily_volatility = 0.012

returns = np.random.normal(
    loc=daily_drift,
    scale=daily_volatility,
    size=n
)
starting_price = 250 
close = starting_price * np.cumprod(1+returns)
spy_data = pd.DataFrame({
    "date": dates,
    "close": close
})

In [14]:
# now lets wrangle our data set
from src.data_prep import realized_volatility_df

df = realized_volatility_df(spy_data)

# using our feature extraction functions
from src.features import ewma, rv_20

df = ewma(df)
df = rv_20(df)
# You may be wondering why drop the na here?
# recall from previously that we recall that we needed to drop na as 
# rv_20 wouldn't have enough data to start populating due to its window
# and then our shifts and calcualtion of percent change during preprocessing would also
# result in nan's (for example shifting the the close level would leave a ga at the end) 
# and then there isn't a percent change at the start of the df 

comparison_data = df.dropna(subset=["RV_20", "EWMA", "r^2_{t+1}", "target_date"])
print(comparison_data[["date", "close", "returns", "RV_20", "EWMA"]].head())
print(comparison_data.shape)

         date       close   returns     RV_20      EWMA
39 2019-02-25  227.204314  0.002662  0.000137  0.000197
40 2019-02-26  229.285869  0.009162  0.000125  0.000191
41 2019-02-27  229.826163  0.002356  0.000125  0.000180
42 2019-02-28  229.576162 -0.001088  0.000125  0.000169
43 2019-03-01  228.815521 -0.003313  0.000111  0.000159
(1787, 8)


In [16]:
# Now lets compare the performance of both features
from src.validation import train_test_folds, run_fold
def walk_forward_validation(start, end, data, feature_col):
    fold_results = []
    print(data.shape)
    print(data["date"].min(), data["date"].max())
    print(data[feature_col].head())
    for test_year, trainingData, testData in train_test_folds(start,end, data):
        fold_results.append(run_fold(trainingData, testData, test_year, feature_col))

    return pd.DataFrame(fold_results)



rv20_results = walk_forward_validation(
    2020,
    2025,
    comparison_data,
    feature_col="RV_20",
)

ewma_results = walk_forward_validation(
    2020,
    2025,
    comparison_data,
    feature_col="EWMA",
)

rv20_results

(1787, 8)
2019-02-25 00:00:00 2025-12-30 00:00:00
39    0.000137
40    0.000125
41    0.000125
42    0.000125
43    0.000111
Name: RV_20, dtype: float64
(1787, 8)
2019-02-25 00:00:00 2025-12-30 00:00:00
39    0.000197
40    0.000191
41    0.000180
42    0.000169
43    0.000159
Name: EWMA, dtype: float64


,test_year,model_mse,baseline_mse,coef,intercept
0,2022,3.102330e-08,3.105671e-08,-0.038004,0.000148
1,2023,4.595454e-08,4.585909e-08,-0.124655,0.000156
2,2024,3.946756e-08,3.955696e-08,-0.061590,0.000151
3,2025,4.154267e-08,4.170237e-08,-0.133162,0.000163


In [12]:
ewma_results

,test_year,model_mse,baseline_mse,coef,intercept
0,2022,3.108344e-08,3.105671e-08,0.032547,0.000138
1,2023,4.591696e-08,4.585909e-08,-0.112662,0.000155
2,2024,3.953067e-08,3.955696e-08,-0.051550,0.000150
3,2025,4.160889e-08,4.170237e-08,-0.091724,0.000157


In [28]:
# Now lets go fold for fold on model behavior
performances = rv20_results[["test_year","model_mse"]].set_index("test_year").join(
    ewma_results[["test_year", "model_mse"]].set_index("test_year"),
    lsuffix='_rv20',
    rsuffix='_ewma'
)
performances

,model_mse_rv20,model_mse_ewma
test_year,,
2022,3.102330e-08,3.108344e-08
2023,4.595454e-08,4.591696e-08
2024,3.946756e-08,3.953067e-08
2025,4.154267e-08,4.160889e-08


In [29]:
# next we can figure out which one did better
performances['model_mse_rv20'] - performances['model_mse_ewma']
# As we can see below it seems that 3 out of 4 times model_mse_rv20 performs better
# considering its mse is smaller than the mse ewma

test_year
2022   -6.014275e-11
2023    3.758533e-11
2024   -6.311051e-11
2025   -6.621598e-11
dtype: float64

In [ ]:
# we can further refine this by expressing the mse square as a relative error to the baseline

# First lets join the baseline mse
performances = performances.join(
    rv20_results.set_index("test_year")[["baseline_mse"]]
)

def improvement_vs_baseline(model_mse, baseline_mse):
    return (baseline_mse - model_mse) / baseline_mse * 100


performances = performances.assign(
    rv20_improvement_pct=lambda df:
        improvement_vs_baseline(
            df["model_mse_rv20"],
            df["baseline_mse"],
        ),

    ewma_improvement_pct=lambda df:
        improvement_vs_baseline(
            df["model_mse_ewma"],
            df["baseline_mse"],
        ),
)

performances

# This tells us that neither feature really gives us that much improvement over the baseline

,model_mse_rv20,model_mse_ewma,rv20_improvement_pct,ewma_improvement_pct,baseline_mse
test_year,,,,,
2022,3.102330e-08,3.108344e-08,0.107581,-0.086074,3.105671e-08
2023,4.595454e-08,4.591696e-08,-0.208140,-0.126182,4.585909e-08
2024,3.946756e-08,3.953067e-08,0.225991,0.066448,3.955696e-08
2025,4.154267e-08,4.160889e-08,0.382945,0.224163,4.170237e-08
